# 04 · RAG Evaluation
### Practical LLM Engineering — Module 04: RAG Systems

> **What you'll learn**
> - The four core RAG metrics: context precision, context recall, faithfulness, answer relevance
> - Implementing RAGAS-style evaluation from scratch
> - LLM-as-a-judge for faithfulness and coherence
> - End-to-end RAG benchmarking pipeline
> - Engineering: building a regression suite for RAG quality

---


## 1. Overview

RAG evaluation requires measuring **four distinct failure modes**:

```
              ┌──────────────────────────────────────────────────┐
              │              RAG Evaluation Framework             │
              │                                                   │
Retrieval  →  │  Context Precision  |  Context Recall            │
              │  (are retrieved     |  (are all relevant          │
              │   chunks relevant?) |   chunks retrieved?)        │
              ├──────────────────────────────────────────────────┤
Generation →  │  Faithfulness       |  Answer Relevance          │
              │  (is the answer     |  (does the answer           │
              │   grounded in ctx?) |   address the question?)    │
              └──────────────────────────────────────────────────┘
```

This maps to the **RAGAS framework** (Es et al., 2023) — a reference-free evaluation approach using LLM judges.


## 2. Setup

In [ ]:
# !pip install numpy matplotlib scipy -q
# ── Shared utilities (carried from Module 03) ────────────────────────
import os, re, json, time, math, random, textwrap
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict
from sklearn.preprocessing import normalize

# ── Mock embedder ─────────────────────────────────────────────────────
class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        return v / max(np.linalg.norm(v), 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts]))

# ── BM25 ──────────────────────────────────────────────────────────────
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self._corpus, self._doc_freqs, self._avgdl, self._N = [], defaultdict(int), 0.0, 0
    def _tok(self, t): return re.findall(r"\b\w+\b", t.lower())
    def fit(self, docs):
        self._corpus = [self._tok(d) for d in docs]; self._N = len(self._corpus)
        self._avgdl  = np.mean([len(d) for d in self._corpus])
        for doc in self._corpus:
            for t in set(doc): self._doc_freqs[t] += 1
    def _idf(self, t):
        n = self._doc_freqs.get(t, 0)
        return math.log((self._N - n + 0.5) / (n + 0.5) + 1)
    def get_scores(self, query):
        qt = self._tok(query)
        scores = []
        for doc in self._corpus:
            dl = len(doc); tf = defaultdict(int)
            for t in doc: tf[t] += 1
            s = sum(self._idf(t) * tf[t] * (self.k1+1) /
                    (tf[t] + self.k1*(1-self.b+self.b*dl/self._avgdl))
                    for t in qt if t in tf)
            scores.append(s)
        return np.array(scores)
    def get_top_k(self, query, k=5):
        s = self.get_scores(query); idx = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in idx]

# ── Vector store ──────────────────────────────────────────────────────
@dataclass
class Doc:
    id: str; text: str; metadata: dict = field(default_factory=dict)

class VectorStore:
    def __init__(self, embedder, dim=64):
        self.embedder = embedder
        self._vecs: np.ndarray = np.zeros((0, dim), dtype=np.float32)
        self._docs: list[Doc]  = []
    def add(self, docs: list[Doc]):
        vecs = self.embedder.embed([d.text for d in docs])
        self._vecs = np.vstack([self._vecs, vecs]) if len(self._vecs) else vecs
        self._docs.extend(docs)
    def search(self, query: str, k=5, where=None) -> list[tuple[Doc, float]]:
        if not self._docs: return []
        q = self.embedder.embed([query])[0]
        scores = self._vecs @ q
        if where:
            for i, doc in enumerate(self._docs):
                for key, val in where.items():
                    if doc.metadata.get(key) != val: scores[i] = -999
        top = np.argsort(-scores)[:k]
        return [(self._docs[i], float(scores[i])) for i in top]
    def __len__(self): return len(self._docs)

# ── Mock LLM ──────────────────────────────────────────────────────────
@dataclass
class LLMResponse:
    text: str; input_tokens: int; output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class MockLLM:
    """Returns templated answers grounded in provided context."""
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def complete(self, system: str, user: str,
                 max_tokens=512, temperature=0.0) -> LLMResponse:
        time.sleep(0.02)
        ctx_match = re.search(r"Context:(.*?)(?:Question:|$)", user, re.DOTALL)
        ctx = ctx_match.group(1).strip()[:200] if ctx_match else ""
        q   = re.search(r"Question:(.*?)$", user, re.DOTALL)
        q   = q.group(1).strip()[:80] if q else user[:80]
        answer = (f"Based on the provided context, {q.lower().rstrip('?')} "
                  f"can be understood as follows: {ctx[:120]}... "
                  f"This answer is grounded in the retrieved documents.")
        n_in  = len((system+user).split())
        n_out = len(answer.split())
        return LLMResponse(answer, n_in, n_out, 45.0)
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

embedder = MockEmbedder(dim=64)
llm      = MockLLM()
print("✅ Shared utilities loaded (embedder + BM25 + VectorStore + MockLLM)")


In [ ]:
# Build a labelled evaluation dataset
@dataclass
class RAGEvalExample:
    question:       str
    ground_truth:   str            # reference answer
    relevant_ids:   set            # ids of truly relevant docs
    metadata:       dict = field(default_factory=dict)

EVAL_DATASET = [
    RAGEvalExample(
        "How does self-attention scale its dot products?",
        "Self-attention divides query-key dot products by the square root of the key dimension (sqrt(d_k)) before applying softmax.",
        {"kb001"},
    ),
    RAGEvalExample(
        "What retrieval algorithm does BM25 use?",
        "BM25 is a sparse retrieval algorithm based on TF-IDF with length normalisation.",
        {"kb005"},
    ),
    RAGEvalExample(
        "How does RAG reduce hallucination?",
        "RAG grounds LLM outputs in retrieved documents, making the model cite specific passages rather than relying on parametric memory.",
        {"kb003"},
    ),
    RAGEvalExample(
        "What is the difference between dense and sparse retrieval?",
        "Dense retrieval uses embedding similarity; sparse retrieval like BM25 uses TF-IDF keyword matching.",
        {"kb005", "kb007"},
    ),
    RAGEvalExample(
        "What is parent-child chunking?",
        "Parent-child chunking stores small chunks for retrieval but returns larger parent documents to the LLM for generation.",
        {"kb013"},
    ),
]

KNOWLEDGE_BASE = [
    Doc("kb001", "Transformers use self-attention with Q·K^T/sqrt(d_k) scaled dot products, then softmax over values.", {"topic":"ml"}),
    Doc("kb002", "BERT is bidirectional; GPT is causal (left-to-right).", {"topic":"ml"}),
    Doc("kb003", "RAG grounds LLM outputs in retrieved documents to reduce hallucination and enable source citation.", {"topic":"rag"}),
    Doc("kb004", "Vector databases index dense embeddings for ANN using HNSW or IVF.", {"topic":"retrieval"}),
    Doc("kb005", "BM25 is sparse TF-IDF retrieval with length normalisation, standard in Elasticsearch.", {"topic":"retrieval"}),
    Doc("kb006", "Chunk size (256-512 tokens) balances embedding quality and retrieval precision.", {"topic":"rag"}),
    Doc("kb007", "Hybrid search fuses dense vector retrieval and sparse BM25 for best recall.", {"topic":"retrieval"}),
    Doc("kb008", "LoRA adds low-rank adapter matrices to fine-tune LLMs with fewer parameters.", {"topic":"ml"}),
    Doc("kb013", "Parent-child chunking retrieves small chunks but returns larger parent context to the LLM.", {"topic":"rag"}),
]
store = VectorStore(embedder); store.add(KNOWLEDGE_BASE)
print(f"Eval dataset: {len(EVAL_DATASET)} examples | KB: {len(store)} docs")


## 3. Retrieval Metrics: Context Precision and Recall

$$
\text{Context Precision@k} = \frac{\text{relevant retrieved chunks}}{\text{total retrieved chunks (k)}}
$$

$$
\text{Context Recall} = \frac{\text{relevant retrieved chunks}}{\text{total relevant chunks}}
$$

These are properties of the **retrieval stage alone** — independent of generation quality.


In [ ]:
def context_precision(retrieved_ids: list[str], relevant_ids: set, k: int = None) -> float:
    ret = retrieved_ids[:k] if k else retrieved_ids
    return sum(1 for i in ret if i in relevant_ids) / max(len(ret), 1)

def context_recall(retrieved_ids: list[str], relevant_ids: set) -> float:
    return len(set(retrieved_ids) & relevant_ids) / max(len(relevant_ids), 1)

def mean_reciprocal_rank(retrieved_ids: list[str], relevant_ids: set) -> float:
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_ids: return 1.0 / rank
    return 0.0


# Evaluate retrieval on every eval example
print("Retrieval evaluation (k=3):\n")
print(f"{'Question':<45} {'Precision':>10} {'Recall':>8} {'MRR':>6}")
print("-" * 72)

prec_list, rec_list, mrr_list = [], [], []
for ex in EVAL_DATASET:
    retrieved    = store.search(ex.question, k=5)
    ret_ids      = [doc.id for doc, _ in retrieved]
    prec         = context_precision(ret_ids, ex.relevant_ids, k=3)
    rec          = context_recall(ret_ids, ex.relevant_ids)
    mrr          = mean_reciprocal_rank(ret_ids, ex.relevant_ids)
    prec_list.append(prec); rec_list.append(rec); mrr_list.append(mrr)
    print(f"{ex.question[:43]:<45} {prec:>10.3f} {rec:>8.3f} {mrr:>6.3f}")

print(f"{'MEAN':<45} {np.mean(prec_list):>10.3f} {np.mean(rec_list):>8.3f} {np.mean(mrr_list):>6.3f}")


## 4. Generation Metrics: Faithfulness and Answer Relevance

**Faithfulness** measures whether every claim in the answer is supported by the retrieved context:

$$
\text{Faithfulness} = \frac{\text{claims supported by context}}{\text{total claims in answer}}
$$

**Answer Relevance** measures whether the answer addresses the question:

$$
\text{Answer Relevance} = \text{sim}(\text{embed}(q), \text{embed}(\text{answer}))
$$


In [ ]:
def faithfulness_score(answer: str, context: str, llm: MockLLM) -> float:
    """
    LLM-based faithfulness: fraction of answer claims supported by context.
    Production: use a stronger judge model and structured output.
    """
    # Extract claims from answer
    system = "You are a fact-checking assistant."
    user   = (f"List the factual claims made in the following answer, one per line.\n\n"
              f"Answer: {answer}")
    resp   = llm(system, user, max_tokens=150)
    claims = [c.strip() for c in resp.text.strip().split("\n") if c.strip()]
    if not claims: return 1.0

    # Check each claim against context
    supported = 0
    for claim in claims:
        check_user = (f"Is the following claim supported by the context? "
                      f"Answer YES or NO.\n\nContext: {context}\n\nClaim: {claim}")
        check_resp = llm("You are a fact-checker.", check_user, max_tokens=5)
        if "yes" in check_resp.text.lower(): supported += 1

    return supported / len(claims)


def answer_relevance_score(question: str, answer: str, embedder) -> float:
    """Embedding cosine similarity between question and answer."""
    vecs = embedder.embed([question, answer])
    return float(np.dot(vecs[0], vecs[1]))


def answer_correctness(answer: str, ground_truth: str, embedder) -> float:
    """Semantic similarity between generated answer and ground truth."""
    vecs = embedder.embed([answer, ground_truth])
    return float(np.dot(vecs[0], vecs[1]))


# ── Full RAGAS-style evaluation ───────────────────────────────────────
RAG_SYSTEM = "You are a factual assistant. Answer using only the context provided. Be concise."

print("Full RAG evaluation (RAGAS-style):\n")
print(f"{'Question':<40} {'Ctx Prec':>9} {'Ctx Rec':>8} {'Faith':>7} {'AnsRel':>8} {'Correct':>9}")
print("-" * 84)

all_metrics = []
for ex in EVAL_DATASET:
    # Retrieve
    retrieved = store.search(ex.question, k=3)
    ret_ids   = [doc.id for doc, _ in retrieved]
    context   = "\n".join(f"[{doc.id}] {doc.text}" for doc, _ in retrieved)

    # Generate
    user   = f"Context:\n{context}\n\nQuestion: {ex.question}\nAnswer:"
    resp   = llm(RAG_SYSTEM, user, max_tokens=150)
    answer = resp.text

    # Metrics
    cp  = context_precision(ret_ids, ex.relevant_ids, k=3)
    cr  = context_recall(ret_ids, ex.relevant_ids)
    fa  = faithfulness_score(answer, context, llm)
    ar  = answer_relevance_score(ex.question, answer, embedder)
    acc = answer_correctness(answer, ex.ground_truth, embedder)

    all_metrics.append({"cp": cp, "cr": cr, "fa": fa, "ar": ar, "acc": acc})
    print(f"{ex.question[:38]:<40} {cp:>9.3f} {cr:>8.3f} {fa:>7.3f} {ar:>8.3f} {acc:>9.3f}")

means = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0]}
print(f"{'MEAN':<40} {means['cp']:>9.3f} {means['cr']:>8.3f} {means['fa']:>7.3f} {means['ar']:>8.3f} {means['acc']:>9.3f}")


## 5. Visualising RAG Quality

In [ ]:
# ── Radar chart: all metrics ─────────────────────────────────────────
metric_names  = ["Context\nPrecision", "Context\nRecall",
                  "Faithfulness", "Answer\nRelevance", "Correctness"]
metric_keys   = ["cp", "cr", "fa", "ar", "acc"]
values        = [means[k] for k in metric_keys]
values_plot   = values + [values[0]]   # close the radar

angles = [n / len(metric_names) * 2 * np.pi for n in range(len(metric_names))]
angles += angles[:1]

fig, (ax_radar, ax_bar) = plt.subplots(1, 2, figsize=(13, 5),
                                         subplot_kw=dict(polar=False))
fig.suptitle("RAG System Evaluation", fontsize=12)

# Bar chart
colors_bar = ["#3498db" if v >= 0.7 else "#f39c12" if v >= 0.5 else "#e74c3c"
               for v in values]
bars = ax_bar.bar(metric_names, values, color=colors_bar, edgecolor="white")
ax_bar.set_ylim(0, 1.05); ax_bar.set_ylabel("Score")
ax_bar.set_title("Mean scores across eval set"); ax_bar.grid(axis="y", alpha=0.3)
ax_bar.axhline(0.7, color="gray", linestyle="--", alpha=0.5, label="0.7 threshold")
ax_bar.legend()
for bar, v in zip(bars, values):
    ax_bar.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.2f}",
                ha="center", fontsize=9, fontweight="bold")

# Per-example heatmap
ax_radar.set_title("Per-example metric heatmap")
mat = np.array([[m[k] for k in metric_keys] for m in all_metrics])
im  = ax_radar.imshow(mat, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax_radar.set_xticks(range(len(metric_keys)))
ax_radar.set_xticklabels(["CP","CR","FA","AR","AC"], fontsize=9)
ax_radar.set_yticks(range(len(EVAL_DATASET)))
ax_radar.set_yticklabels([ex.question[:25]+"..." for ex in EVAL_DATASET], fontsize=7)
plt.colorbar(im, ax=ax_radar, fraction=0.046)

plt.tight_layout(); plt.show()


## 6. RAG Regression Suite

In [ ]:
@dataclass
class RAGEvalRun:
    version:    str
    timestamp:  str
    metrics:    dict
    n_examples: int

    def passes_threshold(self, thresholds: dict) -> bool:
        return all(self.metrics.get(k, 0) >= v for k, v in thresholds.items())


class RAGRegressionSuite:
    """Track RAG quality across pipeline versions and alert on regressions."""
    THRESHOLDS = {"cp": 0.60, "cr": 0.60, "fa": 0.55, "ar": 0.55, "acc": 0.50}

    def __init__(self):
        self.runs: list[RAGEvalRun] = []

    def add(self, run: RAGEvalRun): self.runs.append(run)

    def check(self, run: RAGEvalRun) -> dict:
        regressions = {}
        if self.runs:
            prev_best = {k: max(r.metrics.get(k, 0) for r in self.runs[:-1])
                         for k in run.metrics}
            for k, v in run.metrics.items():
                prev = prev_best.get(k, v)
                if v < prev - 0.05:  # > 5pp drop
                    regressions[k] = {"current": v, "prev_best": prev, "drop": prev - v}
        return regressions

    def plot(self):
        if len(self.runs) < 2: return
        keys  = list(self.runs[0].metrics.keys())
        vers  = [r.version for r in self.runs]
        fig, ax = plt.subplots(figsize=(10, 4))
        for key in keys:
            vals = [r.metrics[key] for r in self.runs]
            ax.plot(vers, vals, "o-", lw=2, label=key, markersize=6)
        for k, thresh in self.THRESHOLDS.items():
            ax.axhline(thresh, linestyle=":", alpha=0.3)
        ax.set_ylabel("Score"); ax.set_xlabel("Pipeline version")
        ax.set_title("RAG Regression Suite — metric history")
        ax.legend(ncol=3, fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
        plt.tight_layout(); plt.show()


# Simulate version history
suite    = RAGRegressionSuite()
versions = [("v1.0", dict(cp=0.60, cr=0.55, fa=0.52, ar=0.58, acc=0.50)),
             ("v1.1", dict(cp=0.68, cr=0.62, fa=0.60, ar=0.63, acc=0.57)),
             ("v1.2", dict(cp=0.72, cr=0.70, fa=0.65, ar=0.68, acc=0.62)),
             ("v1.3", dict(cp=0.65, cr=0.68, fa=0.55, ar=0.60, acc=0.58)),  # regression
             ("v2.0", dict(cp=0.80, cr=0.78, fa=0.74, ar=0.75, acc=0.70))]

for ver, metrics in versions:
    run = RAGEvalRun(ver, f"2025-01-{versions.index((ver,metrics))+1:02d}", metrics, 5)
    suite.add(run)
    reg = suite.check(run)
    status = f"⚠️  REGRESSION {reg}" if reg else "✅ OK"
    print(f"  {ver}: {status}")

suite.plot()


## 7. Key Takeaways

| Metric | What it measures | Ideal value |
|---|---|---|
| **Context Precision** | Fraction of retrieved chunks that are relevant | High (> 0.7) |
| **Context Recall** | Fraction of relevant chunks that are retrieved | High (> 0.7) |
| **Faithfulness** | Fraction of answer claims grounded in context | Very high (> 0.8) |
| **Answer Relevance** | Semantic alignment of answer to question | High (> 0.7) |
| **Answer Correctness** | Similarity to ground-truth answer | Depends on task |

---
**Next notebook →** `05_agentic_rag.ipynb`
